In [0]:
silver_path = "/Volumes/data_architects/silver_layer/preprocessed_data_volume/"
bronze_path = "/Volumes/data_architects/bronze_layer/raw_data_volume/"
df_aisles = spark.read.format("delta").load(bronze_path + "aisles")
df_departments = spark.read.format("delta").load(bronze_path + "departments")
df_orders = spark.read.format("delta").load(bronze_path + "orders")
df_products = spark.read.format("delta").load(bronze_path + "products")
df_order_products = spark.read.format("delta").load(bronze_path + "order_products")

In [0]:
df_aisles.display()


In [0]:
df_departments.display()

In [0]:
from pyspark.sql.functions import col, sum

df_departments.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_departments.columns
]).show()

In [0]:
df_orders.display()

In [0]:
df_orders = df_orders.dropDuplicates()

In [0]:
df_orders.count()

In [0]:
from pyspark.sql.functions import col, sum

df_orders.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_orders.columns
]).show()

In [0]:
df_orders = df_orders.fillna({
    "days_since_prior_order": 0
})

In [0]:
df_products.display()

In [0]:
from pyspark.sql.functions import col, sum

df_products.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_products.columns
]).show()

In [0]:
df_order_products.display()

In [0]:
from pyspark.sql.functions import col, sum

df_order_products.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_order_products.columns
]).show()

In [0]:
df_final = df_order_products \
    .join(df_orders, "order_id", "left") \
    .join(df_products, "product_id", "left") \
    .join(df_aisles, "aisle_id", "left") \
    .join(df_departments, "department_id", "left")

In [0]:
df_final.display()

In [0]:
from pyspark.sql.functions import col, sum

df_final.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_final.columns
]).show()

In [0]:
silver_path = "/Volumes/data_architects/silver_layer/preprocessed_data_volume/"

df_final.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path + "instacart_final")

In [0]:
df_orders.write.format("delta").mode("overwrite").save(silver_path + "orders_clean")

df_products.write.format("delta").mode("overwrite").save(silver_path + "products_clean")

df_order_products.write.format("delta").mode("overwrite").save(silver_path + "order_products_clean")

df_aisles.write.format("delta").mode("overwrite").save(silver_path + "aisles_clean")

df_departments.write.format("delta").mode("overwrite").save(silver_path + "departments_clean")